# Figure S3A Stage clustering

Target manuscript panel: Figure S3A, developmental-stage clustering.

This file retains the complete original joint plotting source. The unchanged
`draw_panels()` function draws both manuscript panels A and B together
(internally labelled `c` and `d`), so this notebook also draws the other panel.
The article output is the all-stage Average/Pearson call in section 12.
All original analysis settings, functions, comparison sections and output
calls are retained without modification.

Reproduce the four accepted marker dotplots: embryonic/all stages × Ward–Euclidean/Average–Pearson. Each figure is saved as a 600-DPI PNG and an editable vector PDF. Only embryonic Ib/Ic groups are merged; postnatal Ia, Ib, Ic and II remain separate. The 53 displayed genes and 31 clustering genes are shared by all four figures.

This downstream notebook starts from the saved integration-stage `Whole_CSS.h5ad`. It rebuilds the current analysis from the original marker counts and the complete common-axis count denominator, without repeating upstream QC or integration.

Run all numbered cells from the project root on first use. A missing cache is rebuilt automatically. For a full rebuild, set `REBUILD_RESULTS=True` and run all cells. Once the cache exists, adjust appearance in a fresh kernel by running **1 → 7 → 8 → the desired figure cell (9–12)**. The only reusable analysis cache is `ROOT`.
The article uses the all-stage Average/Pearson panels in section 12. The other original comparison sections are retained.


## 1. Shared configuration

Set input, output and tool paths at the ROOT placeholders. The cache switch controls analysis rebuilding; figure settings are local to each figure cell. No environment or package inventory is printed.

In [ ]:
#1. =========== Paths and shared settings
from pathlib import Path
from io import BytesIO
import os
import hashlib
import math

# Replace each ROOT placeholder with the required file or directory path before running.
ROOT = Path(".")
if ROOT.name == 'Code_def':
    ROOT = ROOT.parent
# Uses Whole_CSS.h5ad generated by fig1_Whole_integration.ipynb.
INPUT = ROOT
# Set the file or directory path.
RESULT_DIR = ROOT
# Uses hierarchical_clustering.h5ad generated by the hierarchical clustering summary step.
RESULT_FILE = ROOT
# Set the file or directory path.
FIGURE_DIR = ROOT
# Set the file or directory path.
TMP_DIR = ROOT
REBUILD_RESULTS = False           # True rebuilds from the input RNA counts.
rebuild = REBUILD_RESULTS or not RESULT_FILE.exists()
GROUP, STAGE, STUDY = 'source_stage_celltype', 'stage_standardized', 'study_id'
for folder in [RESULT_DIR,FIGURE_DIR,TMP_DIR]:
    folder.mkdir(parents=True,exist_ok=True)
os.environ.update(OMP_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',MKL_NUM_THREADS='1',
                  # Set the file or directory path.
                  MPLCONFIGDIR=str(ROOT),TMP=str(TMP_DIR),TEMP=str(TMP_DIR))

import h5py
import anndata as ad
import numpy as np
import pandas as pd
from anndata.io import read_elem
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator
from matplotlib.text import Text
from matplotlib.backend_bases import RendererBase
from matplotlib.backends.backend_pdf import RendererPdf, Op
from matplotlib import font_manager
from IPython.display import display, Image

STAGE_ORDER = ['E9.5','E10.5','E11.5','E13.5','E14.5','E15.5',
               'E16.5','E17.5','E18.5','P1','P7','P28']
STAGE_COLORS = {'E9.5':'#b08c43', 'E10.5':'#e08bbd', 'E11.5':'#438cb0',
    'E13.5':'#4dc9c9', 'E14.5':'#b06d78', 'E15.5':'#b0946d',
    'E16.5':'#8b92e0', 'E17.5':'#b0b043', 'E18.5':'#55bee0',
    'P1':'#79b043', 'P7':'#e08f55', 'P28':'#9f65c9'}


## 2. Original RNA and fixed marker panel

Changing the marker panel requires rebuilding cells 2–6. Expression is `ln(1 + 10000 × raw marker count / total counts across the 9174 common genes)`. The denominator is not the sum of the displayed markers. Missing measurements remain unavailable, distinct from measured zeros.

In [ ]:
#2. =========== Original marker counts and normalization
TARGET_SUM = 10000

MARKERS = {
    'Early neurogenesis': ['Neurog1','Insm1','Neurod1','Sox2','Hes6','Gadd45g'],
    'SGN differentiation': ['Isl1','Pou4f1','Epha5','Shh','Myt1','Meis2','Prox1','Nhlh1'],
    'Ia / II context': ['Tle4','Gata3','Gfra1','Islr2'],
    'Ib / Ic context': ['Runx1','Ntng1'],
    'Type Ia': ['Calb2','Rxrg','B3gat1','Vsnl1','Cpne9'],
    'Type Ib': ['Calb1','Ttn','Kcnd2','Plcb4','Kcnh2'],
    'Type Ic': ['Lypd1','Grm8','Kcnc2','Sphkap','Pcsk2','Mgat4c','Espn'],
    'Type II': ['Ngfr','Plk5','Th','Calca','Prph','Efna5','Tac1'],
    'Pan-neuronal': ['Tubb3','Nefl'],
    'Exclusion checks': ['Tlx3','Sox10','Plp1','Myo7a','Ptprc','Lyz2','Hba-a1'],
}
DISPLAY_ONLY = {'Pan-neuronal','Exclusion checks'}
genes = [gene for members in MARKERS.values() for gene in members]
gene_groups = np.array([group for group,members in MARKERS.items() for _ in members])

if rebuild:
    input_hash = hashlib.file_digest(INPUT.open('rb'),'sha256').hexdigest()
    with h5py.File(INPUT,'r') as handle:
        obs = read_elem(handle['obs'])
        source_order = [label for label in obs[GROUP].cat.categories if obs[GROUP].eq(label).any()]
        obs = obs[[GROUP,STAGE,STUDY,'library_id','biological_replicate_id']].copy()
        all_genes = handle['uns/css_marker_gene_symbols'].asstr()[:]
        positions = pd.Index(all_genes).get_indexer(genes)
        assert (positions>=0).all(), 'A requested marker is absent from the source marker matrix.'
        raw = handle['obsm/css_marker_raw_counts'][:][:,positions]
        measured = handle['obsm/css_marker_measured'][:][:,positions].astype(bool)
        counts = read_elem(handle['layers/counts'])
    common_gene_count = counts.shape[1]
    denominator = np.asarray(counts.sum(axis=1)).ravel()
    expression = np.log1p(raw*TARGET_SUM/denominator[:,None])
    assert np.array_equal(np.isnan(expression),~measured)
    del counts


## 3. Embryonic-only grouping

Embryonic labels are `Immature Type Ia`, `Immature Type II`, `Immature Ib/Ic candidate` and `Ib/Ic precursor candidate`; the shared `Ia/II precursor candidate` is retained. Postnatal groups are unchanged. Changes to this mapping require rebuilding cells 3–6.

In [ ]:
#3. =========== Embryonic-only grouping
EMBRYONIC_RENAME = {
    'Immature Type Ia SGN': 'Immature Type Ia',
    'Immature Type II SGN': 'Immature Type II',
    'Immature Type Ib SGN': 'Immature Ib/Ic candidate',
    'Immature Type Ic SGN': 'Immature Ib/Ic candidate',
    'Emerging Type Ic': 'Ib/Ic precursor candidate',
    'Type Ib + Type Ic precursor candidate': 'Ib/Ic precursor candidate',
    'Type Ia + Type II precursor candidate': 'Ia/II precursor candidate',
}

if rebuild:
    obs = obs.copy()
    for key in [GROUP,STAGE,STUDY,'library_id','biological_replicate_id']:
        obs[key] = obs[key].astype(object).fillna('').astype(str)
    obs['state_original'] = obs[GROUP].str.split(' | ',regex=False).str[1]
    embryonic = obs[STAGE].str.startswith('E').to_numpy()
    obs['state_merged'] = obs['state_original']
    obs.loc[embryonic,'state_merged'] = obs.loc[embryonic,'state_original'].replace(EMBRYONIC_RENAME)
    obs['group_merged'] = obs[STAGE]+' | '+obs['state_merged']
    np.testing.assert_array_equal(obs.loc[~embryonic,'group_merged'],obs.loc[~embryonic,GROUP])
    valid_replicate = ~obs.biological_replicate_id.str.lower().isin(
        ['', 'not available', 'not_verified', 'not reported', 'not_reported', 'nan'])
    obs['unit'] = obs.biological_replicate_id.where(valid_replicate,obs.library_id)
    type_levels = []
    for label in source_order:
        stage,state = label.split(' | ',1)
        mapped = EMBRYONIC_RENAME.get(state,state) if stage.startswith('E') else state
        merged_label = stage+' | '+mapped
        if merged_label not in type_levels:
            type_levels.append(merged_label)
    panels = {'c':STAGE_ORDER,'d':type_levels}
    levels = panels['c']+panels['d']
    mapping = (obs.groupby([STAGE,GROUP,'group_merged'],sort=False,observed=True)
               .size().rename('n_cells').reset_index())
    mapping['changed'] = mapping[GROUP] != mapping.group_merged


## 4. Weighted expression summaries and shared scaling

Within each group, studies have equal weight, observational units have equal weight within a study, and cells have equal weight within a unit. Known biological replicates define units; otherwise libraries/plates are used. Missing-gene weights are renormalized among measured cells.

All figures reuse one gene-wise Z scale calculated across the full 46 group profiles (`ddof=0`). Colors alone are clipped to [−1, 1]. The same fully measured, nonconstant positive markers are used in every tree; pan-neuronal, exclusion and partial-coverage rows are display-only.

In [ ]:
#4. =========== Group summaries and shared Z scores
DETECTION_THRESHOLD = 0
ZSCORE_DDOF = 0
MIN_SD = 1e-12

if rebuild:
    mean = np.full((len(levels),len(genes)),np.nan)
    fraction,coverage = mean.copy(),mean.copy()
    group_rows = []
    for i,level in enumerate(levels):
        field = STAGE if level in STAGE_ORDER else 'group_merged'
        indices = np.flatnonzero(obs[field].to_numpy() == level)
        selected = obs.iloc[indices]
        unit_n = selected.groupby([STUDY,'unit'],observed=True)['unit'].transform('size')
        study_units = selected.groupby(STUDY,observed=True)['unit'].transform('nunique')
        weights = 1/(unit_n.to_numpy()*study_units.to_numpy()*selected[STUDY].nunique())
        coverage[i] = np.sum(weights[:,None]*measured[indices],axis=0)
        mean[i] = np.divide(np.sum(weights[:,None]*np.nan_to_num(expression[indices]),axis=0),
                            coverage[i],out=np.full(len(genes),np.nan),where=coverage[i]>0)
        fraction[i] = np.divide(np.sum(weights[:,None]*(raw[indices]>DETECTION_THRESHOLD),axis=0),
                                coverage[i],out=np.full(len(genes),np.nan),where=coverage[i]>0)
        group_rows.append(dict(group=level,panel='c' if field==STAGE else 'd',
            stage=selected[STAGE].iloc[0],n_cells=len(indices),n_studies=selected[STUDY].nunique(),
            n_units=len(selected[[STUDY,'unit']].drop_duplicates()),
            studies=';'.join(sorted(selected[STUDY].unique()))))
    group_table = pd.DataFrame(group_rows).set_index('group')
    center,scale = np.nanmean(mean,axis=0),np.nanstd(mean,axis=0,ddof=ZSCORE_DDOF)
    scaled = np.divide(mean-center,scale,out=np.zeros_like(mean),where=scale>MIN_SD)
    scaled[coverage==0] = np.nan
    complete = measured.all(axis=0)
    tree_features = complete & (scale>MIN_SD) & ~np.isin(gene_groups,list(DISPLAY_ONLY))
    scopes = {
        'embryonic':{panel:[label for label in labels if label.split(' | ')[0].startswith('E')]
                     for panel,labels in panels.items()},
        'all':panels,
    }


## 5. Ward and Average hierarchical clustering

Both methods operate on the same 31-gene, unclipped standardized profiles. Ward uses Euclidean geometry; Average uses `1 − Pearson r`. Tree orientation only swaps existing children to minimize postnatal-before-embryonic pairs and then age inversions. It does not change memberships or heights.

References: [SciPy linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html), [correlation distance](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.correlation.html).

In [ ]:
#5. =========== Hierarchical clustering
METHODS = {'ward':('ward','euclidean'),'average':('average','correlation')}
OPTIMAL_ORDER = True

def orientation_cost(order,labels):
    stage = [labels[i].split(' | ')[0] for i in order]
    rank = np.array([STAGE_ORDER.index(s) for s in stage])
    post = np.array([s.startswith('P') for s in stage])
    a,b = np.triu_indices(len(order),1)
    return (int(np.count_nonzero(post[a]&~post[b])),int(np.count_nonzero(rank[a]>rank[b])))

def orient_tree(original,labels):
    # Reorder sibling branches without changing cluster membership or heights.
    tree = original.copy()
    n = len(labels)
    def visit(node):
        if node<n:
            return [node]
        row = node-n
        left,right = map(int,tree[row,:2])
        a,b = visit(left),visit(right)
        if orientation_cost(b+a,labels) < orientation_cost(a+b,labels):
            tree[row,:2] = [right,left]
            return b+a
        return a+b
    visit(2*n-2)
    return tree

if rebuild:
    trees = {}
    for scope,scope_panels in scopes.items():
        for method,(linkage_method,metric) in METHODS.items():
            for panel,panel_levels in scope_panels.items():
                rows = pd.Index(levels).get_indexer(panel_levels)
                data = scaled[rows][:,tree_features]
                original = (linkage(data,method=linkage_method,metric=metric,optimal_ordering=OPTIMAL_ORDER)
                            if method=='ward' else linkage(pdist(data,metric=metric),
                                method=linkage_method,optimal_ordering=OPTIMAL_ORDER))
                trees[(scope,method,panel)] = orient_tree(original,panel_levels)


## 6. Save the minimum reusable result

One compressed H5AD stores the 46 × 53 expression summaries, expression fractions, measurement coverage, unclipped Z values, gene/group order, scaling parameters and eight linkage matrices. Each tree uses the explicitly saved scope/panel group order. The grouping map and relative source provenance are included in the same file; no duplicate TSV/JSON results are needed.

In [ ]:
#6. =========== Store the reusable result
if rebuild:
    gene_table = pd.DataFrame(dict(marker_group=gene_groups,full_source_coverage=complete,
        used_for_tree=tree_features,measured_cell_fraction=measured.mean(axis=0),
        shared_center=center,shared_sd=scale),index=pd.Index(genes,name='gene'))
    group_table.index.name = 'group'
    result = ad.AnnData(X=mean,obs=group_table.copy(),var=gene_table,
        layers={'expression_fraction':fraction,'measurement_coverage':coverage,'shared_z':scaled})
    result.uns['scope_groups'] = {scope:{p:np.asarray(v,dtype=object) for p,v in sp.items()}
                                  for scope,sp in scopes.items()}
    result.uns['linkage'] = {scope:{method:{panel:trees[(scope,method,panel)] for panel in sp}
                                   for method in METHODS} for scope,sp in scopes.items()}
    result.uns['group_mapping'] = mapping
    result.uns['parameters'] = dict(target_sum=TARGET_SUM,common_gene_count=common_gene_count,
        detection_threshold=DETECTION_THRESHOLD,zscore_ddof=ZSCORE_DDOF,min_sd=MIN_SD,
        stage_order=np.asarray(STAGE_ORDER,dtype=object),embryonic_mapping=EMBRYONIC_RENAME,
        methods={m:dict(linkage=a,metric=b) for m,(a,b) in METHODS.items()},
        scaling='Gene-wise Z across the full 46 group profiles; reused in every figure.',
        aggregation='Equal studies, equal units per study, equal cells per unit; renormalize measured-gene weights.')
    result.uns['source'] = dict(file=INPUT.relative_to(ROOT).as_posix(),sha256=input_hash,
        normalization='ln(1 + raw marker counts * 10000 / complete common-axis total counts)')
    result.write_h5ad(RESULT_FILE,compression='gzip')
    print(f'Saved reusable result: {result.n_obs} groups x {result.n_vars} genes.')


## 7. Load reusable plotting data

This is the redraw entry point after cell 1. It needs only the saved H5AD, not the original RNA file or any `tmp` analysis script. Re-run the desired figure cell after changing its appearance parameters.

In [ ]:
#7. =========== Load the reusable result
result = ad.read_h5ad(RESULT_FILE)
group_table = result.obs
levels,genes = result.obs_names.tolist(),result.var_names.tolist()
gene_groups = result.var['marker_group'].astype(str).to_numpy()
complete = result.var['full_source_coverage'].to_numpy()
tree_features = result.var['used_for_tree'].to_numpy()
mean = np.asarray(result.X)
fraction = np.asarray(result.layers['expression_fraction'])
coverage = np.asarray(result.layers['measurement_coverage'])
scaled = np.asarray(result.layers['shared_z'])
scopes = {scope:{panel:list(names) for panel,names in sp.items()}
          for scope,sp in result.uns['scope_groups'].items()}
trees = {(scope,method,panel):np.asarray(tree)
         for scope,methods in result.uns['linkage'].items()
         for method,panels in methods.items() for panel,tree in panels.items()}
orders = {key:leaves_list(tree).tolist() for key,tree in trees.items()}
method_ymax = {method:max(tree[-1,2] for (scope,m,panel),tree in trees.items() if m==method)*1.04
               for method in ['ward','average']}
print(f'Ready: {len(genes)} displayed genes; {int(tree_features.sum())} clustering genes.')


## 8. Shared plotting function and color scale

The colormap, Z range, stage colors and marker order are shared. Per-figure point area, spacing, labels and export DPI are supplied explicitly by cells 9–12. Keep their values synchronized when comparing figures. Each figure saves its accepted 600-DPI PNG and an editable vector PDF (Arial text, inline paths, zero clipping groups); previews use the same figure rendered at 100 DPI.

In [ ]:
#8. =========== Shared figure renderer
LABEL_CONNECTOR_COLOR = '#8A8A8A'
LABEL_CONNECTOR_GAP_PT = 1.2
COLOR_MIN,COLOR_MAX = -1,1
CMAP = LinearSegmentedColormap.from_list('marker_purple',
    ['#F1F1F1','#E8C6DE','#C67ABA','#952E7D','#69005F'])
NORM = Normalize(COLOR_MIN,COLOR_MAX,clip=True)
SHORT_STATES = {
    'O-2: delaminating otic/CVG neuroblast':'Delaminating otic/CVG',
    'O-3: developing CVG neuron (not SGN-specific)':'Developing CVG',
    'Undifferentiated / proliferative CVG':'Proliferative CVG',
}

METHOD_LABELS = {'ward':'Ward / Euclidean','average':'Average / 1 - Pearson r'}


def draw_semantic_text(text,renderer):
    # Keep each multiline annotation in one editable text object.
    if '\n' not in text.get_text():
        return ORIGINAL_TEXT_DRAW(text,renderer)
    lines = []
    def collect(pdf_renderer,gc,x,y,s,prop,angle,ismath=False,mtext=None):
        assert not ismath, 'Multiline PDF annotations must use plain text.'
        copied_gc = pdf_renderer.new_gc()
        copied_gc.copy_properties(gc)
        lines.append((pdf_renderer,copied_gc,x,y,s,prop.copy(),angle))
    original = RendererPdf.draw_text
    try:
        RendererPdf.draw_text = collect
        ORIGINAL_TEXT_DRAW(text,renderer)
    finally:
        RendererPdf.draw_text = original
    if not lines:
        return
    pdf_renderer,gc,*_ = lines[0]
    pdf_renderer.check_gc(gc,gc._rgb)
    pdf_renderer.file.output(Op.begin_text)
    for pdf_renderer,gc,x,y,s,prop,angle in lines:
        font = pdf_renderer._get_font_ttf(prop)
        pdf_renderer.file._character_tracker.track(font,s)
        a = math.radians(angle)
        pdf_renderer.file.output(pdf_renderer.file.fontName(font.fname),
            prop.get_size_in_points(),Op.selectfont,
            math.cos(a),math.sin(a),-math.sin(a),math.cos(a),x,y,Op.textmatrix,
            pdf_renderer.encode_string(s,42),Op.show)
    pdf_renderer.file.output(Op.end_text)


def save_editable_pdf(fig,path,label_tree_pairs):
    # Arial live text; inline vector paths; no per-object clips or raster meshes.
    font_manager.findfont('Arial',fallback_to_default=False)
    for artist in fig.findobj():
        artist.set_clip_on(False)
        artist.set_clip_path(None)
        artist.set_clip_box(None)
        artist.set_rasterized(False)
        if isinstance(artist,Text):
            artist.set_fontfamily('Arial')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    guides = iter(fig.artists)
    for labels,tree_axis in label_tree_pairs:
        for column,text in enumerate(labels):
            box = text.get_window_extent(renderer)
            leaf = tree_axis.transData.transform((10*column+5,0))
            end = [leaf[0],box.y1+LABEL_CONNECTOR_GAP_PT*fig.dpi/72]
            assert end[1]<leaf[1], 'Column label overlaps tree in PDF.'
            start_fig,end_fig = fig.transFigure.inverted().transform([leaf,end])
            next(guides).set_data([start_fig[0],end_fig[0]],[start_fig[1],end_fig[1]])
    collection_draw,marker_draw = RendererPdf.draw_path_collection,RendererPdf.draw_markers
    global ORIGINAL_TEXT_DRAW
    ORIGINAL_TEXT_DRAW = Text.draw
    try:
        Text.draw = draw_semantic_text
        RendererPdf.draw_path_collection = RendererBase.draw_path_collection
        RendererPdf.draw_markers = RendererBase.draw_markers
        with matplotlib.rc_context({'pdf.fonttype':42,'pdf.use14corefonts':False}):
            fig.savefig(path,format='pdf',facecolor='white')
    finally:
        Text.draw = ORIGINAL_TEXT_DRAW
        RendererPdf.draw_path_collection,RendererPdf.draw_markers = collection_draw,marker_draw


def draw_panels(scope,method,filename,appearance):
    gene_indices = np.arange(len(genes))
    PNG_DPI = appearance['PNG_DPI']
    MAX_DOT_AREA = appearance['MAX_DOT_AREA']
    COLUMN_GAP_RATIO = appearance['COLUMN_GAP_RATIO']
    ROW_PITCH = appearance['ROW_PITCH']
    GENE_FONT,TYPE_FONT = appearance['GENE_FONT'],appearance['TYPE_FONT']
    LABEL_CONNECTOR_WIDTH = appearance['LABEL_CONNECTOR_WIDTH']
    scope_panels = scopes[scope]
    actual_diameter = np.sqrt(MAX_DOT_AREA*np.nanmax(fraction))
    pitch = (1+COLUMN_GAP_RATIO)*actual_diameter/72
    width_c,width_d = (len(scope_panels[p])*pitch for p in ['c','d'])
    left,between,right = 1.00,1.40,1.90
    bottom,labels_height,tree_height = 0.65,appearance['LABELS_HEIGHT'],0.80
    matrix_height = len(gene_indices)*ROW_PITCH
    width = left+width_c+between+width_d+right
    height = bottom+matrix_height+labels_height+tree_height+0.90
    fig = plt.figure(figsize=(width,height),dpi=100,facecolor='white')
    top_tree = bottom+matrix_height+labels_height
    checked_text,collections,label_tree_pairs,title_pairs = [],[],[],[]
    for panel,x,matrix_width in [('c',left,width_c),('d',left+width_c+between,width_d)]:
        key = (scope,method,panel)
        ordered_levels = [scope_panels[panel][j] for j in orders[key]]
        rows = pd.Index(levels).get_indexer(ordered_levels)
        pct,cov,z = (a[np.ix_(rows,gene_indices)].T for a in [fraction,coverage,scaled])
        axis = fig.add_axes([x/width,bottom/height,matrix_width/width,matrix_height/height])
        xx,yy = np.meshgrid(np.arange(len(rows)),np.arange(len(gene_indices)))
        positive = (cov>0)&(pct>0)
        points = axis.scatter(xx[positive],yy[positive],s=MAX_DOT_AREA*pct[positive],
                              c=z[positive],cmap=CMAP,norm=NORM,edgecolors='none',linewidths=0)
        collections.append(points)
        missing = cov==0
        axis.scatter(xx[missing],yy[missing],marker='x',s=9,c='0.65',linewidths=0.5)
        axis.set(xlim=(-0.5,len(rows)-0.5),ylim=(len(gene_indices)-0.5,-0.5))
        gene_labels = [genes[j]+('\u2020' if not complete[j] else '') for j in gene_indices]
        axis.set_yticks(np.arange(len(gene_indices)),gene_labels,fontsize=GENE_FONT)
        labels = []
        for label in ordered_levels:
            if panel=='c':
                text = label
            else:
                stage,state = label.split(' | ',1)
                text = f'{stage} {SHORT_STATES.get(state,state)}'
            labels.append(f'{text} (n={group_table.loc[label,"n_cells"]})')
        axis.set_xticks(np.arange(len(rows)),labels,fontsize=TYPE_FONT,
                       rotation=90,ha='center',va='bottom')
        axis.xaxis.tick_top()
        axis.tick_params(length=0,pad=4)
        for spine in axis.spines.values():
            spine.set_visible(False)
        checked_text.extend(axis.get_yticklabels()+axis.get_xticklabels())
        selected_blocks = gene_groups[gene_indices]
        for boundary in np.flatnonzero(selected_blocks[1:]!=selected_blocks[:-1])+1:
            axis.axhline(boundary-0.5,color='0.91',lw=0.5,zorder=0)
        for column,label in enumerate(ordered_levels):
            stage = label.split(' | ')[0]
            axis.add_patch(Rectangle((column-0.5,-0.50),1,0.16,
                facecolor=STAGE_COLORS[stage],edgecolor='none',zorder=0))
        tree_axis = fig.add_axes([x/width,top_tree/height,matrix_width/width,tree_height/height])
        drawn = dendrogram(trees[key],ax=tree_axis,no_labels=True,
                           color_threshold=0,above_threshold_color='#333333')
        assert drawn['leaves']==orders[key]
        tree_axis.set(xlim=(0,10*len(rows)),ylim=(0,method_ymax[method]))
        ticks = MaxNLocator(nbins=3).tick_values(0,method_ymax[method])
        tree_axis.set_yticks(ticks[ticks<=method_ymax[method]])
        tree_axis.tick_params(axis='y',labelsize=6,length=2,pad=2)
        tree_axis.spines[['right','top','bottom']].set_visible(False)
        tree_axis.spines['left'].set_linewidth(0.5)
        tree_axis.set_ylabel('Ward distance' if method=='ward' else 'Cluster distance',fontsize=6,labelpad=3)
        subtitle = 'Developmental stages' if panel=='c' else 'Types and states across ages'
        panel_letter = fig.text((x-0.45)/width,(top_tree+tree_height+0.12)/height,
                                panel,fontsize=15,weight='bold',va='bottom')
        panel_subtitle = fig.text((x+matrix_width/2)/width,(top_tree+tree_height+0.15)/height,
                                  subtitle,fontsize=9,ha='center',va='bottom')
        checked_text.extend([panel_letter,panel_subtitle])
        title_pairs.append((panel_letter,panel_subtitle))
        checked_text.extend(tree_axis.get_yticklabels()+[tree_axis.yaxis.label])
        label_tree_pairs.append((axis.get_xticklabels(),tree_axis))
    legend_x = left+width_c+between+width_d+0.32
    legend_y = bottom+matrix_height
    cax = fig.add_axes([legend_x/width,(legend_y-0.16)/height,1.04/width,0.08/height])
    colorbar = fig.colorbar(ScalarMappable(norm=NORM,cmap=CMAP),cax=cax,
                           orientation='horizontal',ticks=[-1,-.5,0,.5,1])
    colorbar.outline.set_visible(False)
    cax.tick_params(labelsize=6,length=2,pad=2)
    cax.set_title('Avg. Exp. (shared z)',fontsize=7,loc='left',pad=4)
    sizeax = fig.add_axes([legend_x/width,(legend_y-0.82)/height,1.10/width,0.33/height])
    sizeax.set_axis_off()
    sizeax.text(0,1,'% Exp.',fontsize=7,transform=sizeax.transAxes,va='bottom')
    for j,value in enumerate([0,25,50,75,100]):
        sizeax.scatter([j],[.5],s=MAX_DOT_AREA*value/100,color='0.4',edgecolors='none')
        sizeax.text(j,0,str(value),fontsize=6,ha='center',va='top')
    sizeax.set(xlim=(-.4,4.4),ylim=(-.05,1.05))
    method_note = 'Ward linkage\nEuclidean distance' if method=='ward' else 'Average linkage\n1 - Pearson r'
    note = ('Study/unit-balanced\nmeans and proportions\n\n'
        '\u00d7  Not measured\n\u2020  Incomplete full-atlas\n    coverage; display only\n\n'
        f'Tree: {int(tree_features.sum())} complete genes\n{method_note}\nUnclipped shared z\n\n'
        'Shared z: 46 full-atlas\ngroups, reused in all figures\n\n'
        'Ib/Ic merged in embryos\nPostnatal Ib and Ic retained')
    checked_text.append(fig.text(legend_x/width,(legend_y-1.23)/height,note,
                                fontsize=6.5,va='top',linespacing=1.4))
    n_cells = group_table.loc[scope_panels['c'],'n_cells'].sum()
    heading = f'{"Embryonic stages" if scope=="embryonic" else "All stages"} | {METHOD_LABELS[method]} | {n_cells:,} cells'
    checked_text.append(fig.text(.5,(height-.18)/height,heading,fontsize=9,ha='center',va='top'))
    checked_text.append(fig.text(left/width,.22/height,
        'RNA expression. Trees show molecular similarity and do not establish developmental ancestry.',
        fontsize=6.3,va='bottom'))
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    for text in checked_text:
        box = text.get_window_extent(renderer)
        assert fig.bbox.contains(*box.p0) and fig.bbox.contains(*box.p1),f'Clipped text: {text.get_text()}'
    for letter,subtitle in title_pairs:
        assert not letter.get_window_extent(renderer).overlaps(subtitle.get_window_extent(renderer)), 'Panel title overlap'
    for labels,tree_axis in label_tree_pairs:
        boxes = [text.get_window_extent(renderer) for text in labels]
        tree_bottom = tree_axis.get_window_extent(renderer).y0
        assert max(box.y1 for box in boxes)<tree_bottom-2,'Column labels overlap tree'
        assert not any(boxes[j].overlaps(boxes[j+1]) for j in range(len(boxes)-1))
        for column,box in enumerate(boxes):
            leaf = tree_axis.transData.transform((10*column+5,0))
            assert abs(leaf[0]-(box.x0+box.x1)/2)<1e-6
            end = [leaf[0],box.y1+LABEL_CONNECTOR_GAP_PT*fig.dpi/72]
            assert end[1]<leaf[1]
            start_fig,end_fig = fig.transFigure.inverted().transform([leaf,end])
            fig.add_artist(Line2D([start_fig[0],end_fig[0]],[start_fig[1],end_fig[1]],
                transform=fig.transFigure,color=LABEL_CONNECTOR_COLOR,lw=LABEL_CONNECTOR_WIDTH,
                solid_capstyle='butt',clip_on=False,zorder=2))
    for panel,collection in zip(['c','d'],collections):
        diameter_px = np.sqrt(collection.get_sizes().max())*fig.dpi/72
        axis = collection.axes
        coords = axis.transData.transform([[0,0],[1,0],[0,1]])
        col_pitch,row_pitch = abs(coords[1,0]-coords[0,0]),abs(coords[2,1]-coords[0,1])
        assert col_pitch>diameter_px and row_pitch>diameter_px
        assert np.isclose(col_pitch/(actual_diameter*fig.dpi/72)-1,COLUMN_GAP_RATIO)
        boxes = [t.get_window_extent(renderer) for t in axis.get_yticklabels()]
        assert not any(boxes[j].overlaps(boxes[j+1]) for j in range(len(boxes)-1))

    fig.savefig(FIGURE_DIR/filename,dpi=PNG_DPI,facecolor='white')
    preview = BytesIO()
    fig.savefig(preview,format='png',dpi=100,facecolor='white')
    display(Image(data=preview.getvalue(),width=min(850,int(width*100),int(900*width/height))))
    preview.close()
    save_editable_pdf(fig,(FIGURE_DIR/filename).with_suffix('.pdf'),label_tree_pairs)
    plt.close(fig)


## 9. Embryonic stages — Ward + Euclidean

7,355 cells; 9 stages and 22 stage-by-state groups. Only appearance parameters below affect this figure; analysis changes require rebuilding cells 2–6.

In [ ]:
#9. =========== Embryonic stages - Ward + Euclidean
appearance = {
    'PNG_DPI':600,                   # PNG export resolution.
    'MAX_DOT_AREA':36,                # Point area at 100% expression, in pt^2.
    'COLUMN_GAP_RATIO':0.5,           # Gap relative to the largest data dot diameter.
    'ROW_PITCH':0.145,                # Gene-row spacing, in inches.
    'GENE_FONT':7.1,                  # Gene-label font size, in points.
    'TYPE_FONT':6.3,                  # Stage/state-label font size, in points.
    'LABEL_CONNECTOR_WIDTH':0.45,     # Gray leaf-to-label guide width, in points.
    'LABELS_HEIGHT':2.35,             # Vertical space for rotated labels, in inches.
}
# Set the file or directory path.
draw_panels('embryonic','ward',str(ROOT),appearance)


## 10. Embryonic stages — Average + Pearson

7,355 cells; 9 stages and 22 stage-by-state groups. Only appearance parameters below affect this figure; analysis changes require rebuilding cells 2–6.

In [ ]:
#10. =========== Embryonic stages - Average + Pearson
appearance = {
    'PNG_DPI':600,                   # PNG export resolution.
    'MAX_DOT_AREA':36,                # Point area at 100% expression, in pt^2.
    'COLUMN_GAP_RATIO':0.5,           # Gap relative to the largest data dot diameter.
    'ROW_PITCH':0.145,                # Gene-row spacing, in inches.
    'GENE_FONT':7.1,                  # Gene-label font size, in points.
    'TYPE_FONT':6.3,                  # Stage/state-label font size, in points.
    'LABEL_CONNECTOR_WIDTH':0.45,     # Gray leaf-to-label guide width, in points.
    'LABELS_HEIGHT':2.35,             # Vertical space for rotated labels, in inches.
}
# Set the file or directory path.
draw_panels('embryonic','average',str(ROOT),appearance)


## 11. All stages — Ward + Euclidean

8,340 cells; 12 stages and 34 stage-by-state groups. Postnatal Ib and Ic are separate. Only appearance parameters below affect this figure; analysis changes require rebuilding cells 2–6.

In [ ]:
#11. =========== All stages - Ward + Euclidean
appearance = {
    'PNG_DPI':600,                   # PNG export resolution.
    'MAX_DOT_AREA':36,                # Point area at 100% expression, in pt^2.
    'COLUMN_GAP_RATIO':0.5,           # Gap relative to the largest data dot diameter.
    'ROW_PITCH':0.145,                # Gene-row spacing, in inches.
    'GENE_FONT':7.1,                  # Gene-label font size, in points.
    'TYPE_FONT':6.3,                  # Stage/state-label font size, in points.
    'LABEL_CONNECTOR_WIDTH':0.45,     # Gray leaf-to-label guide width, in points.
    'LABELS_HEIGHT':2.35,             # Vertical space for rotated labels, in inches.
}
# Set the file or directory path.
draw_panels('all','ward',str(ROOT),appearance)


## 12. All stages — Average + Pearson

8,340 cells; 12 stages and 34 stage-by-state groups. Postnatal Ib and Ic are separate. Only appearance parameters below affect this figure; analysis changes require rebuilding cells 2–6.

In [ ]:
#12. =========== All stages - Average + Pearson
appearance = {
    'PNG_DPI':600,                   # PNG export resolution.
    'MAX_DOT_AREA':36,                # Point area at 100% expression, in pt^2.
    'COLUMN_GAP_RATIO':0.5,           # Gap relative to the largest data dot diameter.
    'ROW_PITCH':0.145,                # Gene-row spacing, in inches.
    'GENE_FONT':7.1,                  # Gene-label font size, in points.
    'TYPE_FONT':6.3,                  # Stage/state-label font size, in points.
    'LABEL_CONNECTOR_WIDTH':0.45,     # Gray leaf-to-label guide width, in points.
    'LABELS_HEIGHT':2.35,             # Vertical space for rotated labels, in inches.
}
# Set the file or directory path.
draw_panels('all','average',str(ROOT),appearance)
